# 前后端联调与 CORS

## 用两个终端启动前端和后端

后端终端：

~~~bash
fastapi dev
~~~

前端终端：

~~~bash
npm run dev
~~~

## 先让后端数据与前端约定一致

把后端的 profile 补成和前端 site.js 的 home 同构。为了一眼看出数据确实来自后端，可以先在标题后面加一个临时标记：

~~~python
profile = {
    "heroTitle": "关于我（来自后端）",
    "heroSubtitle": "项目，创意，灵感，心得，我的作品",
    "featuredWork": {
        "kicker": "作品",
        "title": "文字实验室",
        "copy": "拼音和情绪，挖掘中文里的细节",
        "linkLabel": "打开作品",
    },
    "identity": {
        "motto": "已识乾坤大，尤怜草木青",
        "learning": "零到全栈",
    },
}
~~~

保存后，用 curl 先确认后端接口本身正常：

~~~bash
curl http://localhost:8000/api/profile
~~~

能看到完整 JSON 和“来自后端”的标记，说明接口和数据结构已经准备好。

## 用现成的前端代码替换

用 zero-to-tech-5-5/ 里的文件覆盖项目同名文件

| 文件 | 改动 |
| --- | --- |
| HomeView.jsx | 变成客户端组件；先用 site.js 数据打底，再 GET /api/profile，拿到后端数据后更新界面 |
| TextLabView.jsx | 变成客户端组件；把分析结果状态提升到这里，再传给输入卡和结果卡 |
| InputCard.jsx | 点击开始分析时，把文字 POST 给 /api/analyze |
| ResultCard.jsx | 显示父组件传来的结果，没有结果时显示默认占位 |
| css/lab.css | 新增 .lab-error 样式，请求失败时显示提示 |

先记住几点：

- 因为要在浏览器中发请求，HomeView、InputCard 等组件顶部会有 "use client"。
- 这些组件里的后端地址暂时是 http://localhost:8000，最后会把它移入配置文件。
- 请求失败时，代码用 try/catch 做基本兜底：主页失败就保持 site.js 的打底数据，输入卡失败就在按钮上方提示。

替换完后，保存并打开 http://localhost:3000。按预期，主页标题应该变成“关于我（来自后端）”。如果仍然显示原来的“关于我”，不要急着改代码，先按数据流排查。

## 第一次撞墙：顺着线索排查

遇到“后端接口正常，但网页没有更新”，不要盯着代码猜。一次请求会在三个地方留下线索：

| 看哪里 | 回答什么问题 |
| --- | --- |
| 浏览器 Network | 请求发出去了吗？ |
| 后端终端 | 后端收到了吗？又是怎么处理的？ |
| 浏览器 Console | 结果为什么没有交到 JavaScript 手里？ |

### 第一站：浏览器 Network

打开开发者工具，切到 Network，刷新页面，找到 profile。如果能看到这条请求，说明请求确实发出去了，不是 fetch 代码根本没有执行。

开发环境中偶尔看到两条相同的 GET 也不用慌。Next.js App Router 默认开启 React 严格模式，开发时可能多执行一次 Effect 来检查副作用；正式构建不会因为这个检查多发一次请求。

### 第二站：后端终端

回到后端终端，应该能看到类似：

~~~text
GET /api/profile HTTP/1.1 200 OK
~~~

这说明请求已经到达后端，而且后端处理完并返回了 200。从后端视角看，请求是成功的。

### 第三站：浏览器 Console(控制台)

切到 Console，如果看到类似红字：

~~~text
Access to fetch at 'http://localhost:8000/api/profile'
from origin 'http://localhost:3000' has been blocked by CORS policy:
No 'Access-Control-Allow-Origin' header is present...
~~~

把三站线索连起来：

- Network 有请求，说明请求发出去了。
- 后端日志返回 200，说明请求收到了并处理成功。
- curl 也能拿到 JSON，说明接口本身没坏。
- 只有网页 JavaScript 读不到结果。

问题不在接口是否运行，也不在路径是否写错，而在浏览器的 CORS 规则。

## CORS 到底拦了什么

CORS 是 Cross-Origin Resource Sharing 的缩写，中文叫“跨源资源共享”，是浏览器对网页 JavaScript 的一条安全规则。

### 什么叫“跨源”

一个源（origin）由三部分组成：

~~~text
协议 + 域名 + 端口
~~~

任何一项不同，就是不同的源：

~~~text
http://localhost:3000
http://localhost:8000
~~~

这两个地址的协议都是 http，域名都是 localhost，但端口不同，因此是两个源。网页脚本从 3000 访问 8000，就是跨源请求。

### 请求其实已经到达后端

对刚才这个简单 GET 来说，浏览器已经把请求发给后端，后端也返回了 200。CORS 拦下的不是“请求到达服务器”，而是：

> 浏览器不允许当前网页的 JavaScript 读取这份未经授权的跨源响应。

这也是 curl 一直畅通的原因：CORS 是浏览器对网页脚本的规则，curl 不是网页，不受这条规则约束。

### 浏览器怎样询问，后端怎样回答

网页发起跨源请求时，浏览器会自动带上：

~~~text
Origin: http://localhost:3000
~~~

后端如果愿意让这个来源读取响应，就在响应头中给出：

~~~text
Access-Control-Allow-Origin: http://localhost:3000
~~~

请求里说明来源，响应里给出许可；浏览器看到两边对得上，才把响应交给网页 JavaScript。

## 给 FastAPI 加上 CORS

要让网页读到响应，就要让后端带上 Access-Control-Allow-Origin。与其给每个接口都手写响应头，不如使用中间件统一处理。

中间件可以理解为加在“请求进入、响应离开”必经路径上的一层处理，所有请求和响应都会经过它。

打开 main.py，增加导入：

~~~python
from fastapi.middleware.cors import CORSMiddleware
~~~

紧跟在 app = FastAPI() 后加入：

~~~python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
)
~~~

allow_origins 控制 Access-Control-Allow-Origin 这一行响应头。把前端地址填进去，后端就允许这个来源的网页脚本读取响应。

保存后后端会自动重启。刷新主页，标题应该变成“关于我（来自后端）”，数据终于从 8000 端口流到了 3000 端口的网页。

再看 Network 里的 profile，Response Headers 中应该多出：

~~~text
access-control-allow-origin: http://localhost:3000
~~~

确认成功后，可以把标题中的“（来自后端）”临时标记删掉，恢复正常文案。

## 第二次撞墙：POST 被预检拦下

主页的 GET 通了，接下来试文字实验室的开始分析。界面出现 Failed to fetch，继续到现场找线索。

Network 中点第二个 analyze ，显示了没有在代码中写过的 OPTIONS 请求，而且它失败了；真正想发的 POST 反而没有出现：

Console 可能提示：

~~~text
Method POST is not allowed by Access-Control-Allow-Methods in preflight response.
~~~

又是 CORS，但和第一次不同：

- 第一次：GET 已经发出、后端也返回了，只是响应没有许可，网页不能读取。
- 这一次：POST 在发出之前就被 OPTIONS 拦住了。

### 这个 OPTIONS 是浏览器自动发的

这个 OPTIONS 叫 CORS 预检（preflight）。OPTIONS：“我能对这个资源做什么？”。

为什么简单 GET 没有预检，而这个 POST 有？

浏览器把跨源请求分为两类：

- **简单请求**：方法普通、请求头也普通，例如主页的 GET。浏览器直接发，之后检查响应是否有许可。
- **非简单请求**：例如 POST。浏览器会先询问后端，再决定是否发送真正的请求。

预检会询问：

1. 这个来源允许吗？
2. 允许使用 POST 吗？
3. 允许携带这些请求头吗？

预检通过后才发送真正的 POST；预检不通过，POST 根本不会离开浏览器。

### 预检问的，正是我们没回答的

Console 已经说明原因：Method POST is not allowed。

回头看 CORS 中间件，我们只写了 allow_origins，只说明“谁能来”，没有说明“能用什么方法”。预检由 CORSMiddleware 自动应答，它发现 POST 不在允许列表中，于是拒绝预检。不需要为 OPTIONS 单独写函数。

补上 allow_methods：

~~~python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["GET", "POST"],
)
~~~

allow_methods 应该如实写项目需要的跨源方法。当前项目用到 GET 和 POST，所以写这两个即可。

预检还可能询问“能不能带某些请求头”，对应参数是 allow_headers。本例的 Content-Type 属于浏览器通常放行的请求头，不必专门声明；将来如果请求要携带自定义 token 等头部，再把它们列入 allow_headers。

保存后端、等待自动重启，回到 Network 点击“开始分析”，点击 analyze 会看到 POST 200

## 最后一步：把写死的地址收进配置

GET 和 POST 都正常后，还留着一个问题：前端组件里写死了 http://localhost:8000。

### 创建 .env.local

在项目根目录，新建 .env.local：

~~~dotenv
NEXT_PUBLIC_API_BASE_URL=http://localhost:8000
~~~

NEXT_PUBLIC_ 前缀表示这个值允许进入浏览器端代码。因此它只能放后端地址这类可以公开的配置，不能放 API 密钥、密码等秘密信息。任何带 NEXT_PUBLIC_ 的值都可能被打包到浏览器，等于公开。

在 HomeView.jsx 和 InputCard.jsx 的 import 下方加入：

~~~javascript
const API = process.env.NEXT_PUBLIC_API_BASE_URL;
~~~

把硬编码地址改为配置变量：

~~~javascript
const res = await fetch("http://localhost:8000/api/profile");

# 将http：……改为`${API}/api/profile`
const res = await fetch(`${API}/api/profile`);
~~~

前端和后端的配置原则是对称的：后端 CORS 中的 http://localhost:3000 同样是环境相关配置，正式项目中也应集中管理。本节先把前端这一侧讲透，后端配置在部署时再统一处理。

### 修改环境变量后必须重启前端

.env.local 是在开发服务器启动时读取的，保存文件还不够，必须重启前端：

~~~text
按 Ctrl + C 停止原来的开发服务器
~~~

然后重新运行：

~~~bash
npm run dev
~~~

重启后，再分别验证主页的 GET 和文字实验室的 POST，应该都能正常工作。

项目的 .gitignore 通常已经包含 .env*，所以 .env.local 默认不会进入 Git。环境变量不提交的原因是不同环境需要不同地址，并且其中可能包含敏感值；部署时再由服务器或平台注入对应配置。

## 见证：网站真正活了

把一次 POST 请求的完整链路再看一遍：

~~~text
输入文字
  → 浏览器发送 POST
  → OPTIONS 预检通过
  → FastAPI 接收并校验请求体
  → Python 计算结果
  → FastAPI 返回 JSON
  → 前端拿到 JSON、更新界面
  → 结果区自动刷新
~~~